* 정상치 데이터만을 이용해 학습하고 건조기 고장을 탐지하는 scikit-learn 기반의 파이썬 코드입니다.
* LLE 단독으로는 새로운 데이터에 대한 이상치를 직접 반환하지 않으므로 다음과 같은 순서로 진행합니다.
  * 오직 정상 데이터만으로 LLE의 공간(매니폴드)을 학습
  * 정상 데이터가 축소된 영역을 기반으로 One-Class SVM을 학습

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

In [ ]:
# 1. 가상의 건조기 데이터 생성
np.random.seed(42)

# [정상 데이터 200개] - 오직 이 데이터만 모델 학습(fit)에 사용합니다.
n_normal_train = 200
t_train = np.linspace(0, 4 * np.pi, n_normal_train)
train_temp = 50 + 20 * np.sin(t_train) + np.random.normal(0, 1.0, n_normal_train)
train_humid = 55 + 35 * np.cos(t_train) + np.random.normal(0, 1.0, n_normal_train)
train_vib = 0.3 + 0.1 * np.sin(t_train*2) + np.random.normal(0, 0.01, n_normal_train)
train_curr = 8 + 3 * np.cos(t_train/2) + np.random.normal(0, 0.1, n_normal_train)
X_train_normal = np.column_stack([train_temp, train_humid, train_vib, train_curr])

# [테스트용 정상 데이터 50개]
n_normal_test = 50
t_test = np.linspace(4 * np.pi, 5 * np.pi, n_normal_test)
test_temp = 50 + 20 * np.sin(t_test) + np.random.normal(0, 1.0, n_normal_test)
test_humid = 55 + 35 * np.cos(t_test) + np.random.normal(0, 1.0, n_normal_test)
test_vib = 0.3 + 0.1 * np.sin(t_test*2) + np.random.normal(0, 0.01, n_normal_test)
test_curr = 8 + 3 * np.cos(t_test/2) + np.random.normal(0, 0.1, n_normal_test)
X_test_normal = np.column_stack([test_temp, test_humid, test_vib, test_curr])

# [테스트용 고장 데이터 20개]
n_fault_test = 20
fault_temp = np.random.uniform(75, 85, n_fault_test)
fault_humid = np.random.uniform(60, 70, n_fault_test)
fault_vib = np.random.uniform(0.4, 0.5, n_fault_test)
fault_curr = np.random.uniform(11, 13, n_fault_test)
X_test_fault = np.column_stack([fault_temp, fault_humid, fault_vib, fault_curr])

In [ ]:
# 2. 데이터 표준화 (정상 학습 데이터의 스케일을 기준으로 전체 적용)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_normal) # 정상 데이터로만 기준(fit) 설정
X_test_norm_scaled = scaler.transform(X_test_normal)
X_test_fault_scaled = scaler.transform(X_test_fault)

In [ ]:
# 3. LLE 모델 구축 및 학습 (정상 데이터로만 fit)
# n_neighbors는 주변 이웃의 수입니다.
lle = LocallyLinearEmbedding(n_neighbors=15, n_components=2, random_state=42)
X_train_lle = lle.fit_transform(X_train_scaled) # 정상 데이터 매니폴드 학습

# LLE 변환기를 이용해 테스트 데이터(정상/고장)를 저차원으로 투영(transform)
X_test_norm_lle = lle.transform(X_test_norm_scaled)
X_test_fault_lle = lle.transform(X_test_fault_scaled)

In [ ]:
# 4. One-Class SVM 모델 구축 및 학습 (정상 LLE 좌표로만 fit)
# nu: 이상치 비율 허용치, gamma: 커널 반경
oc_svm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
oc_svm.fit(X_train_lle) # 정상 영역 정의

In [ ]:
# 5. 테스트 데이터 진단 (1: 정상, -1: 고장)
pred_norm = oc_svm.predict(X_test_norm_lle)
pred_fault = oc_svm.predict(X_test_fault_lle)

In [ ]:
# 6. 결과 시각화
plt.figure(figsize=(12, 6))

# 학습에 사용된 정상 데이터 (배경)
plt.scatter(X_train_lle[:, 0], X_train_lle[:, 1], 
            c='lightgray', alpha=0.5, label='Train Normal (Reference)')

# 검증용 정상 데이터
plt.scatter(X_test_norm_lle[:, 0], X_test_norm_lle[:, 1], 
            c='blue', alpha=0.7, edgecolor='k', label='Test Normal')

# 검증용 고장 데이터
plt.scatter(X_test_fault_lle[:, 0], X_test_fault_lle[:, 1], 
            c='red', alpha=0.8, edgecolor='k', marker='X', s=100, label='Test Fault')

plt.title('Semi-Supervised Fault Detection (LLE + One-Class SVM)')
plt.xlabel('LLE Component 1')
plt.ylabel('LLE Component 2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 7. 평가 출력
print(f"[진단 결과]")
print(f"새로운 정상 데이터 {n_normal_test}개 중 {np.sum(pred_norm == 1)}개를 '정상'으로 올바르게 판정")
print(f"새로운 고장 데이터 {n_fault_test}개 중 {np.sum(pred_fault == -1)}개를 '고장'으로 정확하게 탐지")